# DREAM LegNet for yeast promoter design

[Notebook](dream-legnet-promoter.ipynb) · [Installation](../docs/installation.md)

Optimize an 80-nt yeast promoter insert for predicted activity. Keep the assay context fixed and exclude runs of six or more identical bases.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

import re
from pathlib import Path

import torch
from bioseq_models import load_model
from MotifSift import call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
dream = load_model("dream_legnet", variable_region_only=True, device=device, batch_size=2048)
suffix = "GGTTACGGCTGTT"
context = dream.encoder(["A" * 80 + suffix])

class PromoterScorer:
    task_names = dream.task_names
    device = dream.device

    def score(self, sequences):
        return dream.score([sequence + suffix for sequence in sequences])

    def score_soft(self, probabilities):
        batch = len(probabilities)
        fixed = context.to(probabilities).expand(batch, -1, -1)
        insert = probabilities[:, :, [0, 2, 1, 3]].transpose(1, 2)
        dna = torch.cat((fixed[:, :4, :57], insert, fixed[:, :4, -13:]), dim=2)
        inputs = torch.cat((dna, fixed[:, 4:, :]), dim=1)
        return dream.model(inputs).reshape(-1, 1)

model = PromoterScorer()
run = Path("outputs/dream_legnet_promoter")




## 2. Generate candidates



In [2]:
def no_homopolymer6(sequences):
    return [re.search(r"([ACGT])\1{5}", sequence) is None for sequence in sequences]

seeds = random_seed_sequences(80, 100, seed=42)
designs = generate_designs(
    model, length=80, initial=seeds, objective="maximize",
    constraint=no_homopolymer6, output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,16.377661,16.505711
basesearch,100,16.153419,16.698112
beam_search,100,16.381478,16.913494
cross_entropy_method,100,16.052849,16.533146
den,100,15.722754,15.942613
fast_seqprop,100,16.086090,16.477905
genetic_algorithm,100,16.382809,16.671318
ism_greedy,100,16.293449,16.827705
ledidi,100,16.246962,16.804264




## 3. Discover motifs



In [3]:
motifs = call_motifs(designs, model, output_dir=run / "motifs")
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-NRGTATAWAAAVG,270,270,433.0
1,2-RKCCGGGTAACC,97,97,780.0
3,6-CGCGCGAA,12,12,48.0
14,24-GCAAATTTTTCCCA,12,12,135.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,CGAACACCTTACAGGTCCTGAGGCGAAAAAGTATATAAATCGCTCA...,16.913494,1-NRGTATAWAAAVG
94,2,GGAATCGCAATTACCCGGACGAAGGAAGATTCTCGACAACGGAACG...,16.551338,2-RKCCGGGTAACC;24-GCAAATTTTTCCCA
163,3,CGCGCCAGATCGCGCGAAACGGTCCGGACATTTTACGGCAAAAAGT...,16.455391,6-CGCGCGAA
173,4,GTCGTCGGCGGCCACCCGCGCGAAAATTTTACTCGAAATGATAGCC...,16.449242,1-NRGTATAWAAAVG;6-CGCGCGAA;24-GCAAATTTTTCCCA
426,5,AACAGTGTGCCGAGCGTCGTGAAGAAAATTGTCCGGGTGTCACTAC...,16.251534,1-NRGTATAWAAAVG;2-RKCCGGGTAACC
561,6,GGCCTATTTGCCTGTCCGGCCGGGCCGCGCGAAATTAGAACTAGCT...,16.140270,2-RKCCGGGTAACC;6-CGCGCGAA
292,7,ATTTTATACGTATTACGTCGTTACCCTACCTAAGAGGAAAAATTTT...,16.362518,24-GCAAATTTTTCCCA
825,8,GTTTTCCGTCCGATGGTGACCGAAGGGTAATGGAGGGAGTGATTAT...,15.975979,2-RKCCGGGTAACC
9,9,CATTACCTAATTAGGTAAATGGTGAGGTGCACCCGGCCTATATAAA...,16.824574,1-NRGTATAWAAAVG;24-GCAAATTTTTCCCA
38,10,CGCGCGAAGGTCCAGACGCCCGTAATGCGAACCCGAAGCGTGATCG...,16.671318,1-NRGTATAWAAAVG;6-CGCGCGAA
